In [2]:
import sys
import os
from os.path import join
import glob
from copy import deepcopy

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from IPython.display import display, clear_output

sys.path.insert(0, "../../ABC-SN/code")
import abcsn_config
import abcsn_training
import data_degrading as dg
import data_plotting as dplt
import data_preparation as dp
import preprocessing

sys.path.insert(0, "../code")
import review_spectrum as rs
import spectral_features as sf
import measure_signal as ms

from icecream import ic
from importlib import reload

reload(rs)
reload(sf)
reload(ms)

REPO_DIR = "../"
DATA_DIR = join(REPO_DIR, "data")
BIANCO_DENOISING_DIR = join(DATA_DIR, "original_resolution_parquet/bianco_denoising")

In [3]:
df_SNRmetadata = rs.load_SNRmetadata()
subtype_mask = df_SNRmetadata["SN Subtype"] == "Ic-broad"
df_SNRmetadata[subtype_mask].info()

<class 'pandas.core.frame.DataFrame'>
Index: 228 entries, 1448 to 3634
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype   
---  ------               --------------  -----   
 0   SN Name              228 non-null    category
 1   SN Subtype           228 non-null    category
 2   SN Subtype ID        228 non-null    int64   
 3   SN Maintype          228 non-null    category
 4   SN Maintype ID       228 non-null    int64   
 5   Spectral Phase       228 non-null    float64 
 6   Denoising Parameter  89 non-null     float64 
 7   minima_i             8 non-null      float64 
 8   searchBlu            89 non-null     float64 
 9   searchRed            89 non-null     float64 
 10  useBlu               89 non-null     object  
 11  useRed               89 non-null     object  
 12  maxBlu               89 non-null     float64 
 13  maxRed               89 non-null     float64 
 14  noiseWindowBlu       89 non-null     float64 
 15  noiseWindowRed       89 

In [28]:
df_data, df_metadata, wvl = rs.load_sn_data()
df_SNRmetadata = rs.load_SNRmetadata()

mask = df_metadata["SN Name"] == "sn2013dx"
mask &= df_metadata["Spectral Phase"] == 4.6
df_metadata[mask]

,SN Name,SN Subtype,SN Subtype ID,SN Maintype,SN Maintype ID,Spectral Phase
3563,sn2013dx,Ic-broad,8,Ic,2,4.6
3564,sn2013dx,Ic-broad,8,Ic,2,4.6


In [29]:
inds = df_metadata[mask].index.to_numpy()

df_metadata.drop(axis=0, index=inds, inplace=True)
df_data.drop(axis=0, index=inds, inplace=True)
df_SNRmetadata.drop(axis=0, index=inds, inplace=True)

df_metadata.reset_index(inplace=True, drop=True)
df_data.reset_index(inplace=True, drop=True)
df_SNRmetadata.reset_index(inplace=True, drop=True)

In [30]:
df_metadata.loc[inds[0]-1:inds[-1]+1]

,SN Name,SN Subtype,SN Subtype ID,SN Maintype,SN Maintype ID,Spectral Phase
3562,sn2013dx,Ic-broad,8,Ic,2,2.3
3563,sn2013dx,Ic-broad,8,Ic,2,6.4
3564,sn2013dx,Ic-broad,8,Ic,2,10.7
3565,sn2013dx,Ic-broad,8,Ic,2,13.4


In [31]:
df_data.loc[inds[0]-1:inds[-1]+1]

,2501.69,2505.08,2508.48,2511.87,2515.28,2518.69,2522.1,2525.51,2528.94,2532.36,...,9872.21,9885.59,9898.98,9912.39,9925.82,9939.27,9952.73,9966.21,9979.71,9993.24
3562,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3563,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3564,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3565,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [32]:
df_SNRmetadata.loc[inds[0]-1:inds[-1]+1]

,SN Name,SN Subtype,SN Subtype ID,SN Maintype,SN Maintype ID,Spectral Phase,Denoising Parameter,minima_i,searchBlu,searchRed,useBlu,useRed,maxBlu,maxRed,noiseWindowBlu,noiseWindowRed
3562,sn2013dx,Ic-broad,8,Ic,2,2.3,30.0,3.0,500.0,500.0,True,True,3.0,0.0,100.0,100.0
3563,sn2013dx,Ic-broad,8,Ic,2,6.4,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN
3564,sn2013dx,Ic-broad,8,Ic,2,10.7,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN
3565,sn2013dx,Ic-broad,8,Ic,2,13.4,NaN,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN


In [33]:
df_metadata.to_parquet("../data/forSNR/metadata_nodupe.parquet")
df_data.to_parquet("../data/forSNR/data_nodupe.parquet")
df_SNRmetadata.to_parquet("../data/forSNR/SNRmetadata_nodupe.parquet")